# 00 Daten

Kern-Daten (Chip, Emulator, FP32) laden, prüfen und zu Tabellen je Bild aufbereiten. Alle weiteren Notebooks lesen nur diese Tabellen.
Plan: Vault-Notiz „Auswertungsplan“ (Tag `auswertungsplan-v1`). Logik: `thesis.analysis.kern`, `thesis.analysis.maps`.

In [1]:
import numpy as np
import pandas as pd
from thesis.analysis import kern

N_JOBS = 4                     # parallele Prozesse, je Prozess bis ≈ 3 GB RAM
SPEC, CACHE = kern.KernSpec(), kern.CACHE_DIR
kern.provenance()

{'commit': 'bf7a082',
 'dirty': False,
 'dirty_files': [],
 'notebooks_geändert': ['notebooks/00_daten.ipynb',
  'notebooks/tabellen_manifest.json'],
 'plan_tag': '65a6639',
 'zeit': '2026-11-24 18:12'}

## 1 Inventar

**Was:** alle Run-Ordner des Kerns aus `results/*/meta.json` einlesen.
**Warum:** Grundlage für Prüfung und Extraktion; zeigt auf einen Blick, ob jede Quelle in voller Blockzahl da ist.

In [2]:
inv = kern.inventory(SPEC)
inv.pivot_table(index="source", columns="model", values="n", aggfunc=["count", "sum"])

count                         sum         
model       mobilenet_v3_large resnet18 mobilenet_v3_large resnet18
source                                                             
chip_k01                     6        6              48000    48000
chip_k02                     6        6              48000    48000
chip_k03                     6        6              48000    48000
chip_k04                     6        6              48000    48000
chip_k05                     6        6              48000    48000
chip_k06                     6        6              48000    48000
chip_k07                     6        6              48000    48000
chip_k08                     6        6              48000    48000
chip_k09                     6        6              48000    48000
chip_k10                     6        6              48000    48000
emu_k01                      2        2               5000     5000
emu_k02                      2        2               5000     5000
emu_k03                      2        2               5000     5000
emu_k04                      2        2               5000     5000
emu_k05                      2        2               5000     5000
emu_k06                      2        2               5000     5000
emu_k07                      2        2               5000     5000
emu_k08                      2        2               5000     5000
emu_k09                      2        2               5000     5000
emu_k10                      2        2               5000     5000
emuvoll_k01                 16       16              48000    48000
fp32                         8        8              48000    48000
fp32_teil                    1        1               5000     5000

## 2 Prüfung

**Was:** Blöcke lückenlos, alle Dateien da, Bild-IDs = Liste, kein Lauf `dirty`, HEF-Hash je Compile gleich (Chip = Emulator), Emulator-Zielklassen aus dem richtigen FP32-Lauf.
**Warum:** Läufe mit Fehlern dürfen nicht in die Auswertung (Plan Abschn. 8).

In [3]:
checks = kern.check_inventory(inv, SPEC)
print(f"{checks.ok.sum()}/{len(checks)} Prüfungen bestanden")
checks[~checks.ok]

148/148 Prüfungen bestanden


,prüfung,ok,detail


In [4]:
assert checks.ok.all(), "Prüfung fehlgeschlagen – nicht weitermachen"

## 3 Extraktion

**Was:** jeden Block einmal lesen und speichern: 7×7-Rohkarten (Eigen-CAM, Grad-CAM), Ausgabegrößen je Bild (Top-1/5, Logit-Cosinus, ΔP-Basis, Ziel-Logit) und den Aktivierungsabstand Emulator ↔ Chip (TF2).
**Warum:** Die Rohdaten sind ≈ 36 GB; danach arbeiten alle Notebooks mit wenigen hundert MB. Wiederaufnehmbar: fertige Blöcke werden übersprungen.

In [5]:
%%time
kern.extract_all(inv, SPEC, CACHE, n_jobs=N_JOBS)

übersprungen 20261116-214346_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261116-221027_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261116-223720_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261116-230412_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261116-233050_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261116-235737_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261117-002415_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261117-005111_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261116-212123_roosevelt_mobilenet_v3_large_fp32_imagenet
übersprungen 20261116-210112_roosevelt_resnet18_fp32_imagenet
übersprungen 20261116-210345_roosevelt_resnet18_fp32_imagenet
übersprungen 20261116-210617_roosevelt_resnet18_fp32_imagenet
übersprungen 20261116-210849_roosevelt_resnet18_fp32_imagenet
übersprungen 20261116-211119_roosevelt_resnet18_fp32_imagenet
übersprungen 20261116-211350_roosevelt_res

## 4 Konsistenz

**Was:** (a) FP32 auf der Teilmenge (eigener Lauf) gegen FP32 auf allen 48 000 für dieselben Bilder; (b) Emulator k01 auf der Teilmenge gegen den vollen Emulator-Lauf k01.
**Warum:** Dieselben Bilder wurden zweimal gerechnet. Erwartet: gleich oder nur Gleitkomma-Rauschen (anderer Batch). Sonst ist unklar, welche Fassung gilt.

In [6]:
def gleich(model, name_a, name_b):
    A, B = kern.load_source(CACHE, model, name_a), kern.load_source(CACHE, model, name_b)
    pos = {i: n for n, i in enumerate(A["ids"])}
    ia = np.array([pos[i] for i in B["ids"]])
    zeile = {"model": model, "vergleich": f"{name_a} ↔ {name_b}", "n": len(ia)}
    for key in ("eig_raw", "grad_raw", "logits", "logit_t"):
        if key in A and key in B:
            zeile[f"max|Δ| {key}"] = float(np.abs(A[key][ia] - B[key]).max())
    if "top1" in A and "top1" in B:
        zeile["Top-1 gleich"] = float((A["top1"][ia] == B["top1"]).mean())
    if "logits" in A and "logits" in B:
        zeile["Top-1 gleich"] = float((A["logits"][ia].argmax(1) == B["logits"].argmax(1)).mean())
    return zeile

pd.DataFrame([gleich(m, "fp32", "fp32_teil") for m in SPEC.models]
             + [gleich(m, "emuvoll_k01", "emu_k01") for m in SPEC.models])

,model,vergleich,n,max|Δ| eig_raw,max|Δ| grad_raw,max|Δ| logits,Top-1 gleich,max|Δ| logit_t
0,resnet18,fp32 ↔ fp32_teil,5000,0.0,0.000000,0.0,1.0,NaN
1,mobilenet_v3_large,fp32 ↔ fp32_teil,5000,0.0,0.000000,0.0,1.0,NaN
2,resnet18,emuvoll_k01 ↔ emu_k01,5000,0.0,0.000179,NaN,1.0,0.0
3,mobilenet_v3_large,emuvoll_k01 ↔ emu_k01,5000,0.0,0.000000,NaN,1.0,0.0


## 5 Tabellen

**Was:** je Modell drei Tabellen in `auswertung_cache/tabellen/`:
`quellen` (Quelle × Bild: Vorhersage, Ausgabegrößen, Sättigung, leere/konstante Karte),
`vergleiche` (Paar × Bild: Top-15/20-IoU, CC, ausgerichtet `*_al`, Spearman/KLD bei FP32 ↔ X),
`aktivierung` (Emulator ↔ Chip je Bild: relativer Fehler, Cosinus, Anteil bitgleich).
**Warum:** Grundlage für 01–06. Die Definitionen folgen dem Plan (Abschn. 3–5, 8): Eigen-CAM ohne FP32-Ausrichtung, Grad-CAM-Metriken leer (NaN), wenn eine Karte leer ist.

In [7]:
%%time
pfade = kern.compute_tables(SPEC, CACHE, n_jobs=N_JOBS)
pfade

CPU times: user 106 ms, sys: 47.5 ms, total: 154 ms
Wall time: 225 ms


{'quellen_resnet18': PosixPath('/Users/dbenner/hm-inf/sem7/Bachelorarbeit/bachelor-thesis-code/auswertung_cache/tabellen/quellen_resnet18.parquet'),
 'aktivierung_resnet18': PosixPath('/Users/dbenner/hm-inf/sem7/Bachelorarbeit/bachelor-thesis-code/auswertung_cache/tabellen/aktivierung_resnet18.parquet'),
 'vergleiche_resnet18': PosixPath('/Users/dbenner/hm-inf/sem7/Bachelorarbeit/bachelor-thesis-code/auswertung_cache/tabellen/vergleiche_resnet18.parquet'),
 'quellen_mobilenet_v3_large': PosixPath('/Users/dbenner/hm-inf/sem7/Bachelorarbeit/bachelor-thesis-code/auswertung_cache/tabellen/quellen_mobilenet_v3_large.parquet'),
 'aktivierung_mobilenet_v3_large': PosixPath('/Users/dbenner/hm-inf/sem7/Bachelorarbeit/bachelor-thesis-code/auswertung_cache/tabellen/aktivierung_mobilenet_v3_large.parquet'),
 'vergleiche_mobilenet_v3_large': PosixPath('/Users/dbenner/hm-inf/sem7/Bachelorarbeit/bachelor-thesis-code/auswertung_cache/tabellen/vergleiche_mobilenet_v3_large.parquet')}

In [8]:
for model in SPEC.models:
    q = pd.read_parquet(pfade[f"quellen_{model}"])
    v = pd.read_parquet(pfade[f"vergleiche_{model}"])
    a = pd.read_parquet(pfade[f"aktivierung_{model}"])
    falsch = int((q.grad_target_ok == False).sum())          # Grad-CAM-Ziel ≠ FP32-Top-1, erwartet 0
    print(f"{model}: quellen {len(q):,} · vergleiche {len(v):,} · aktivierung {len(a):,} · Grad-CAM-Ziel falsch: {falsch}")
    display(v.assign(b_art=v.b.astype(str).str.split("_").str[0])
             .groupby(["cam", "a", "b_art"], observed=True).image_id.agg(["size", "nunique"]))

resnet18: quellen 626,000 · vergleiche 3,384,000 · aktivierung 98,000 · Grad-CAM-Ziel falsch: 0


size  nunique
cam  a        b_art                   
eig  chip_k01 chip     432000    48000
              emu        5000     5000
              emuvoll   48000    48000
     chip_k02 chip     384000    48000
              emu        5000     5000
     chip_k03 chip     336000    48000
              emu        5000     5000
     chip_k04 chip     288000    48000
              emu        5000     5000
     chip_k05 chip     240000    48000
              emu        5000     5000
     chip_k06 chip     192000    48000
              emu        5000     5000
     chip_k07 chip     144000    48000
              emu        5000     5000
     chip_k08 chip      96000    48000
              emu        5000     5000
     chip_k09 chip      48000    48000
              emu        5000     5000
     chip_k10 emu        5000     5000
     emu_k01  emu       45000     5000
     emu_k02  emu       40000     5000
     emu_k03  emu       35000     5000
     emu_k04  emu       30000     5000
     emu_k05  emu       25000     5000
     emu_k06  emu       20000     5000
     emu_k07  emu       15000     5000
     emu_k08  emu       10000     5000
     emu_k09  emu        5000     5000
     fp32     chip     480000    48000
              emu       50000     5000
              emuvoll   48000    48000
grad emu_k01  emu       45000     5000
     emu_k02  emu       40000     5000
     emu_k03  emu       35000     5000
     emu_k04  emu       30000     5000
     emu_k05  emu       25000     5000
     emu_k06  emu       20000     5000
     emu_k07  emu       15000     5000
     emu_k08  emu       10000     5000
     emu_k09  emu        5000     5000
     fp32     emu       50000     5000
              emuvoll   48000    48000

mobilenet_v3_large: quellen 626,000 · vergleiche 3,384,000 · aktivierung 98,000 · Grad-CAM-Ziel falsch: 0


size  nunique
cam  a        b_art                   
eig  chip_k01 chip     432000    48000
              emu        5000     5000
              emuvoll   48000    48000
     chip_k02 chip     384000    48000
              emu        5000     5000
     chip_k03 chip     336000    48000
              emu        5000     5000
     chip_k04 chip     288000    48000
              emu        5000     5000
     chip_k05 chip     240000    48000
              emu        5000     5000
     chip_k06 chip     192000    48000
              emu        5000     5000
     chip_k07 chip     144000    48000
              emu        5000     5000
     chip_k08 chip      96000    48000
              emu        5000     5000
     chip_k09 chip      48000    48000
              emu        5000     5000
     chip_k10 emu        5000     5000
     emu_k01  emu       45000     5000
     emu_k02  emu       40000     5000
     emu_k03  emu       35000     5000
     emu_k04  emu       30000     5000
     emu_k05  emu       25000     5000
     emu_k06  emu       20000     5000
     emu_k07  emu       15000     5000
     emu_k08  emu       10000     5000
     emu_k09  emu        5000     5000
     fp32     chip     480000    48000
              emu       50000     5000
              emuvoll   48000    48000
grad emu_k01  emu       45000     5000
     emu_k02  emu       40000     5000
     emu_k03  emu       35000     5000
     emu_k04  emu       30000     5000
     emu_k05  emu       25000     5000
     emu_k06  emu       20000     5000
     emu_k07  emu       15000     5000
     emu_k08  emu       10000     5000
     emu_k09  emu        5000     5000
     fp32     emu       50000     5000
              emuvoll   48000    48000

## 6 Plausibilität FP32

**Was:** FP32-Top-1/Top-5 auf den 48 000 Bildern.
**Warum:** bestätigt Labels und Bildzuordnung. Soll (Logbuch 17.11.2026): ResNet18 69,89 %, MobileNetV3 74,08 %; torchvision auf 50 000: 69,758 % bzw. 74,042 %. Hier nur FP32, keine INT8-Ergebnisse.

In [9]:
pd.concat([pd.read_parquet(pfade[f"quellen_{m}"]).query("source == 'fp32'") for m in SPEC.models]) \
  .groupby("model")[["top1_ok", "top5_ok"]].mean().mul(100).round(2)

,top1_ok,top5_ok
model,,
mobilenet_v3_large,74.08,91.39
resnet18,69.89,89.08
